# Task 4: Visual Search

In [20]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from PIL import Image, ImageOps
from torchvision import transforms

In [21]:
# Works whether Jupyter starts in the repository root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "preprocessed_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "preprocessed_datasets" / "train" / "styles_train.csv"
IMAGE_DIR = PROJECT_ROOT / "preprocessed_datasets" / "train" / "images_train"
SPLIT_DIR = PROJECT_ROOT / "splits" / "task4"

df = pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37745 entries, 0 to 37744
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  37745 non-null  int64
 1   gender              37745 non-null  str  
 2   masterCategory      37745 non-null  str  
 3   subCategory         37745 non-null  str  
 4   articleType         37745 non-null  str  
 5   baseColour          37745 non-null  str  
 6   season              37745 non-null  str  
 7   year                37745 non-null  int64
 8   usage               37745 non-null  str  
 9   productDisplayName  37745 non-null  str  
dtypes: int64(2), str(8)
memory usage: 2.9 MB


In [22]:
stratify_cols = [
    "articleType",
    "gender",
]

# One binary feature per category across all chosen columns
stratify_features = pd.get_dummies(
    df[stratify_cols].astype(str),
    prefix=stratify_cols,
)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42,
)

train_idx, test_idx = next(splitter.split(df, stratify_features))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train:", train_df.shape)
print("Test: ", test_df.shape)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

Train: (33968, 10)
Test:  (3777, 10)


In [19]:
# Supervised metric-learning label: article type within its gender group.
# The mapping is fitted on the training partition only.
LABEL_COLUMN = "articleType_gender"
LABEL_ID_COLUMN = "articleType_gender_id"

for split_df in (train_df, test_df):
    split_df[LABEL_COLUMN] = (
        split_df["articleType"].str.strip() + "__" + split_df["gender"].str.strip()
    )

# JSON is portable and avoids serialising executable pickle/joblib objects.
classes = sorted(train_df[LABEL_COLUMN].unique().tolist())
label_to_index = {label: index for index, label in enumerate(classes)}
train_df[LABEL_ID_COLUMN] = train_df[LABEL_COLUMN].map(label_to_index).astype("int64")
test_df[LABEL_ID_COLUMN] = test_df[LABEL_COLUMN].map(label_to_index).astype("Int64")

unseen_test_labels = sorted(set(test_df[LABEL_COLUMN]) - set(label_to_index))
if unseen_test_labels:
    print(
        f"Warning: {len(unseen_test_labels)} test label(s) are absent from training and have <NA> IDs."
    )
    print(unseen_test_labels)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task4"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_ENCODER_PATH = ARTIFACT_DIR / "articleType_gender_label_encoder.json"
with LABEL_ENCODER_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "label_column": LABEL_COLUMN,
            "label_id_column": LABEL_ID_COLUMN,
            "fit_split": "train",
            "separator": "__",
            "classes": classes,
            "label_to_index": label_to_index,
        },
        file,
        indent=2,
        sort_keys=True,
    )

class_counts = train_df[LABEL_COLUMN].value_counts()
print(f"Saved {len(classes)} classes")
print(f"Classes with fewer than 2 training examples: {(class_counts < 2).sum()}")

['Innerwear Vests__Women', 'Shirts__Girls', 'Tracksuits__Women']
Saved 249 classes
Classes with fewer than 2 training examples: 30


## Image preprocessing

The images in `preprocessed_datasets` are assumed to have already been standardised: orientation corrected and converted to RGB. They are letterbox-resized to the shared ResNet-18 input size of 128x128: the aspect ratio is preserved and unused space is padded white.

RGB normalisation is fitted only on the training split. Augmentation is used only for training; validation, test, gallery, and query images always use the deterministic evaluation transform.

In [12]:
RESNET_INPUT_SIZE = (128, 128)


class LetterboxResize:
    """Resize to fit within a canvas, padding instead of stretching or cropping."""

    def __init__(self, size, fill=(255, 255, 255)):
        self.size = tuple(size)
        self.fill = fill

    def __call__(self, image):
        return ImageOps.pad(
            image.convert('RGB'),
            self.size,
            method=Image.Resampling.BILINEAR,
            color=self.fill,
            centering=(0.5, 0.5),
        )

letterbox_to_resnet = LetterboxResize(RESNET_INPUT_SIZE)

def compute_rgb_mean_std(record_ids):
    """Calculate per-channel RGB statistics using training images only."""
    channel_sum = torch.zeros(3, dtype=torch.float64)
    channel_sum_sq = torch.zeros(3, dtype=torch.float64)
    pixel_count = 0

    for record_id in record_ids:
        path = IMAGE_DIR / f'{int(record_id)}.jpg'

        if not path.exists():
            raise FileNotFoundError(f'Missing image: {path}')
        
        with Image.open(path) as image:
            tensor = transforms.ToTensor()(letterbox_to_resnet(image)).to(torch.float64)

        channel_sum += tensor.sum(dim=(1, 2))
        channel_sum_sq += (tensor ** 2).sum(dim=(1, 2))
        pixel_count += tensor.shape[1] * tensor.shape[2]

    mean = channel_sum / pixel_count
    std = torch.sqrt(channel_sum_sq / pixel_count - mean ** 2)
    return mean.float().tolist(), std.float().tolist()

train_mean, train_std = compute_rgb_mean_std(train_df['id'])
print('Training RGB mean:', np.round(train_mean, 4))
print('Training RGB std: ', np.round(train_std, 4))

Training RGB mean: [0.8862 0.8742 0.8695]
Training RGB std:  [0.2397 0.2513 0.2546]


In [15]:
image_preprocessing_config = {
    "input_color_mode": "RGB",
    "resize": {
        "method": "letterbox",
        "target_size": [128, 128],
        "interpolation": "bilinear",
        "padding_color_rgb": [255, 255, 255],
        "centering": [0.5, 0.5],
    },
    "tensor": {
        "layout": "CHW",
        "dtype": "float32",
        "value_range_before_normalization": [0.0, 1.0],
    },
    "normalization": {
        "mean_rgb": train_mean,
        "std_rgb": train_std,
    },
}

with (ARTIFACT_DIR / "image_preprocessing.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(image_preprocessing_config, file, indent=2)

In [14]:
# Triplet Margin, SupCon, Multi-Similarity, and ArcFace training.
# Mild geometric augmentation preserves the full product and its colour cues.
metric_train_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        fill=(255, 255, 255),
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# Use unchanged for validation, test, gallery, and query images.
metric_eval_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std),
])

# CAE input and MSE reconstruction target are letterboxed to 128x128 in [0, 1].
# Pair this with a decoder ending in Sigmoid().
cae_transform = transforms.Compose([
    letterbox_to_resnet,
    transforms.ToTensor(),
])

### Transform assignment

- **CAE:** use `cae_transform` for both input and reconstruction target; start without augmentation.
- **Triplet Margin, Multi-Similarity, ArcFace:** use `metric_train_transform` during training and `metric_eval_transform` otherwise.
- **SupCon:** apply `metric_train_transform` twice independently to each training image for its two views; use `metric_eval_transform` for evaluation.

Random resized crops, strong rotations, and strong colour jitter are intentionally excluded because they can remove product details or distort colour information relevant to visual search.

## Five-model training pipeline setup

Everything above this point is the original preprocessing pipeline. The cells below prepare an inner development split from the outer training partition; the outer test partition is not used for model development.


In [ ]:
import copy
import math
import random
import time
from collections import Counter

import faiss
import matplotlib.pyplot as plt
import optuna
import seaborn as sns
import torch.nn.functional as F
from pytorch_metric_learning import losses
from torch import nn
from torch.utils.data import BatchSampler, DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm


In [ ]:
SEED = 42
INPUT_SIZE = RESNET_INPUT_SIZE
MIN_CLASS_SIZE = 5
VAL_FRACTION = 0.10
CLASSES_PER_BATCH = 16
IMAGES_PER_CLASS = 4
CAE_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 256
NUM_WORKERS = 0

RUN_SMOKE_TESTS = False
RUN_OPTUNA = False
RUN_FINAL_TRAINING = False
RUN_GALLERY_EXPORT = False

N_TRIALS = 20
TUNING_EPOCHS = 10
FINAL_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 8
MIN_DELTA = 1e-4
OPTUNA_DB = ARTIFACT_DIR / "optuna.db"

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
PIN_MEMORY = AMP_ENABLED
print("Device:", DEVICE, "| Mixed precision:", AMP_ENABLED)


## Inner model-development split

Rare composite classes are filtered only for the five-model experiment. The original outer training and test dataframes remain available as outer_train_df and test_df.


In [ ]:
outer_train_df = train_df.copy()

outer_class_counts = outer_train_df[LABEL_COLUMN].value_counts()
eligible_labels = set(
    outer_class_counts[outer_class_counts >= MIN_CLASS_SIZE].index
)
development_df = outer_train_df[
    outer_train_df[LABEL_COLUMN].isin(eligible_labels)
].copy()

inner_stratify_columns = ["articleType", "gender"]
inner_stratify_features = pd.get_dummies(
    development_df[inner_stratify_columns].astype(str),
    prefix=inner_stratify_columns,
)
inner_splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_FRACTION,
    random_state=SEED,
)
inner_train_idx, inner_val_idx = next(
    inner_splitter.split(development_df, inner_stratify_features)
)
train_df = development_df.iloc[inner_train_idx].copy()
val_df = development_df.iloc[inner_val_idx].copy()

In [ ]:
overlap = set(train_df["id"]) & set(val_df["id"])
if overlap:
    raise ValueError(f"Inner split contains {len(overlap)} overlapping IDs")
if val_df[LABEL_ID_COLUMN].isna().any():
    raise ValueError("Inner validation contains an unmapped label")

print(f"Model train/validation: {len(train_df):,}/{len(val_df):,}")
print("Excluded rare rows:", len(outer_train_df) - len(development_df))
print("Test rows kept aside:", len(test_df))

In [ ]:
metric_preprocessing_config = copy.deepcopy(image_preprocessing_config)
cae_preprocessing_config = copy.deepcopy(image_preprocessing_config)
cae_preprocessing_config.pop("normalization", None)

## Datasets and batching

SupCon requests two independent augmented views. Other models receive one image tensor per row.


In [ ]:
class FashionImageDataset(Dataset):
    def __init__(self, frame, image_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = self.image_dir / f"{int(row['id'])}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            output = self.transform(image)

        label = row.get(LABEL_ID_COLUMN, pd.NA)
        label = -1 if pd.isna(label) else int(label)
        return {"image": output, "label": label, "id": int(row["id"])}


In [ ]:
class PKBatchSampler(BatchSampler):
    def __init__(self, labels, classes_per_batch, images_per_class, seed):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.classes_per_batch = classes_per_batch
        self.images_per_class = images_per_class
        self.seed = seed
        self.epoch = 0
        self.class_indices = {
            label: np.flatnonzero(self.labels == label)
            for label in np.unique(self.labels)
        }
        self.batch_size = classes_per_batch * images_per_class
        self.batch_count = max(1, len(self.labels) // self.batch_size)

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return self.batch_count

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        available_classes = np.array(sorted(self.class_indices))

        for _ in range(self.batch_count):
            chosen_classes = rng.choice(
                available_classes,
                size=self.classes_per_batch,
                replace=False,
            )
            batch = []
            for label in chosen_classes:
                choices = self.class_indices[label]
                selected = rng.choice(choices, self.images_per_class, replace=False)
                batch.extend(selected.tolist())
            rng.shuffle(batch)
            yield batch


In [ ]:
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def standard_loader(dataset, batch_size, shuffle=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=NUM_WORKERS > 0,
    )

In [ ]:
def make_two_view_transform(transform):
    def apply(image):
        return transform(image), transform(image)

    return apply


def make_training_loader(model_name):
    transform = cae_transform if model_name == "cae" else metric_train_transform
    if model_name == "supcon":
        transform = make_two_view_transform(transform)
    dataset = FashionImageDataset(train_df, IMAGE_DIR, transform)
    if model_name == "cae":
        return standard_loader(dataset, CAE_BATCH_SIZE, shuffle=True)

    sampler = PKBatchSampler(
        train_df[LABEL_ID_COLUMN],
        CLASSES_PER_BATCH,
        IMAGES_PER_CLASS,
        SEED,
    )
    return DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        worker_init_fn=seed_worker,
        persistent_workers=NUM_WORKERS > 0,
    )


In [ ]:
def make_evaluation_loader(frame, model_name):
    transform = cae_transform if model_name == "cae" else metric_eval_transform
    ordered_frame = frame.sort_values("id").reset_index(drop=True)
    dataset = FashionImageDataset(ordered_frame, IMAGE_DIR, transform)
    return standard_loader(dataset, EVAL_BATCH_SIZE, shuffle=False)


gallery_parts = []
for _, group in train_df.groupby(LABEL_COLUMN, sort=True):
    sample_size = min(20, len(group))
    gallery_parts.append(group.sample(sample_size, random_state=SEED))

tuning_gallery_df = pd.concat(gallery_parts).sort_values("id").reset_index(drop=True)
print("Tuning gallery rows:", len(tuning_gallery_df))


In [ ]:
def make_model_loaders(model_name):
    return {
        "train": make_training_loader(model_name),
        "gallery": make_evaluation_loader(tuning_gallery_df, model_name),
        "query": make_evaluation_loader(val_df, model_name),
    }

## Shared ResNet-18 encoder

The model factory resets seed 42 immediately before constructing each encoder. Trained weights are never transferred between objectives.


In [ ]:
class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        network = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(network.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, images):
        feature_map = self.features(images)
        embedding = self.pool(feature_map).flatten(1)
        return F.normalize(embedding, p=2, dim=1)


In [ ]:
def build_cae_decoder():
    channels = [512, 256, 128, 64, 32]
    blocks = []

    for input_channels, output_channels in zip(channels[:-1], channels[1:]):
        blocks.extend([
            nn.ConvTranspose2d(
                input_channels, output_channels, kernel_size=4, stride=2, padding=1
            ),
            nn.BatchNorm2d(output_channels),
            nn.ReLU(inplace=True),
        ])

    blocks.extend([
        nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
        nn.Sigmoid(),
    ])
    return nn.Sequential(*blocks)


In [ ]:
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = build_cae_decoder()

    def forward(self, images):
        feature_map = self.encoder.features(images)
        embedding = self.encoder.pool(feature_map).flatten(1)
        reconstruction = self.decoder(feature_map)
        embedding = F.normalize(embedding, p=2, dim=1)
        return reconstruction, embedding


## Model and loss factory


In [ ]:
MODEL_NAMES = ("cae", "triplet", "supcon", "multi_similarity", "arcface")


def build_model_and_loss(model_name, parameters):
    set_seed(SEED)
    encoder = ResNet18Encoder()

    if model_name == "cae":
        return ConvolutionalAutoencoder(encoder), nn.MSELoss()
    if model_name == "triplet":
        loss = losses.TripletMarginLoss(parameters.get("margin", 0.3))
        return encoder, loss
    if model_name == "supcon":
        loss = losses.SupConLoss(parameters.get("temperature", 0.1))
        return encoder, loss
    if model_name == "multi_similarity":
        loss = losses.MultiSimilarityLoss(
            alpha=parameters.get("alpha", 2.0),
            beta=parameters.get("beta", 50.0),
            base=parameters.get("base", 0.5),
        )
        return encoder, loss
    if model_name == "arcface":
        encoder.arcface_loss = losses.ArcFaceLoss(
            num_classes=len(classes),
            embedding_size=512,
            margin=math.degrees(parameters.get("margin", 0.3)),
            scale=parameters.get("scale", 32),
        )
        return encoder, encoder.arcface_loss
    raise ValueError(f"Unknown model: {model_name}")


## Training helpers


In [ ]:
def compute_training_loss(model_name, model, loss_function, batch):
    labels = batch["label"].to(DEVICE, non_blocking=True)

    if model_name == "supcon":
        first_view, second_view = batch["image"]
        images = torch.cat([first_view, second_view], dim=0).to(DEVICE)
        embeddings = model(images)
        return loss_function(embeddings, labels.repeat(2))

    images = batch["image"].to(DEVICE, non_blocking=True)
    if model_name == "cae":
        reconstruction, _ = model(images)
        return loss_function(reconstruction, images)
    if model_name == "arcface":
        return model.arcface_loss(model(images), labels)

    embeddings = model(images)
    return loss_function(embeddings, labels)


In [ ]:
def train_one_epoch(model_name, model, loss_function, loader, optimizer, scaler, epoch):
    model.train()
    if hasattr(loader.batch_sampler, "set_epoch"):
        loader.batch_sampler.set_epoch(epoch)

    running_loss = 0.0
    example_count = 0

    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            loss = compute_training_loss(model_name, model, loss_function, batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = len(batch["label"])
        running_loss += loss.detach().item() * batch_size
        example_count += batch_size

    return running_loss / max(1, example_count)


## Embedding extraction and retrieval metrics


In [ ]:
def retrieval_embeddings(model, images):
    outputs = model(images)
    embeddings = outputs[1] if isinstance(outputs, tuple) else outputs
    return F.normalize(embeddings, dim=1)


def extract_embeddings(model, loader):
    model.eval()
    embeddings, identifiers, labels = [], [], []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                batch_embeddings = retrieval_embeddings(model, images)
            embeddings.append(batch_embeddings.cpu())
            identifiers.append(batch["id"].cpu())
            labels.append(batch["label"].cpu())

    return (
        torch.cat(embeddings).numpy().astype("float32"),
        torch.cat(identifiers).numpy().astype("int64"),
        torch.cat(labels).numpy().astype("int64"),
    )


In [ ]:
def build_faiss_index(gallery_embeddings):
    embeddings = np.ascontiguousarray(gallery_embeddings, dtype="float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


def search_cosine(query_embeddings, gallery_embeddings, maximum_k=10):
    index = build_faiss_index(gallery_embeddings)
    search_k = min(maximum_k, len(gallery_embeddings))
    scores, indices = index.search(
        np.ascontiguousarray(query_embeddings, dtype="float32"),
        search_k,
    )
    return scores, indices, index


In [ ]:
def retrieval_metrics(ranked_indices, query_labels, gallery_labels, ks=(1, 5, 10)):
    gallery_counts = Counter(gallery_labels.tolist())
    metrics = {}
    max_available_k = ranked_indices.shape[1]

    for requested_k in ks:
        k = min(requested_k, max_available_k)
        precisions, recalls = [], []

        for row, query_label in zip(ranked_indices[:, :k], query_labels):
            hits = gallery_labels[row] == query_label
            hit_count = float(np.asarray(hits).sum())
            precisions.append(hit_count / k)
            recalls.append(hit_count / max(1, gallery_counts[int(query_label)]))

        metrics[f"Precision@{requested_k}"] = float(np.mean(precisions))
        metrics[f"Recall@{requested_k}"] = float(np.mean(recalls))

    metrics["mAP@10"] = mean_average_precision_at_k(
        ranked_indices,
        query_labels,
        gallery_labels,
        10,
    )
    return metrics


In [ ]:
def mean_average_precision_at_k(ranked_indices, query_labels, gallery_labels, k):
    gallery_counts = Counter(gallery_labels.tolist())
    effective_k = min(k, ranked_indices.shape[1])
    average_precisions = []

    for row, query_label in zip(ranked_indices[:, :effective_k], query_labels):
        relevant = (gallery_labels[row] == query_label).astype(np.float32)
        cumulative_precision = np.cumsum(relevant) / np.arange(1, effective_k + 1)
        denominator = min(gallery_counts[int(query_label)], effective_k)
        average_precision = float((cumulative_precision * relevant).sum())
        average_precisions.append(average_precision / max(1, denominator))

    return float(np.mean(average_precisions))


In [ ]:
def evaluate_retrieval(model, gallery_loader, query_loader):
    gallery_embeddings, _, gallery_labels = extract_embeddings(
        model,
        gallery_loader,
    )
    query_embeddings, _, query_labels = extract_embeddings(
        model,
        query_loader,
    )
    _, ranked_indices, _ = search_cosine(
        query_embeddings,
        gallery_embeddings,
        maximum_k=10,
    )
    return retrieval_metrics(ranked_indices, query_labels, gallery_labels)


## Early stopping and multi-epoch fitting


In [ ]:
class EarlyStopping:
    def __init__(self, patience, minimum_delta):
        self.patience = patience
        self.minimum_delta = minimum_delta
        self.best_score = -math.inf
        self.bad_epochs = 0

    def update(self, score):
        improved = score > self.best_score + self.minimum_delta
        self.best_score = score if improved else self.best_score
        self.bad_epochs = 0 if improved else self.bad_epochs + 1
        return improved

    @property
    def should_stop(self):
        return self.bad_epochs >= self.patience


def cpu_state_dict(model):
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    }


In [ ]:
def new_optimizer_and_scheduler(model, parameters):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=parameters["learning_rate"],
        weight_decay=parameters["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    return optimizer, scheduler


In [ ]:
def fit_model(model_name, model, loss_function, loaders, parameters, max_epochs, trial=None):
    optimizer, scheduler = new_optimizer_and_scheduler(model, parameters)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    stopper = EarlyStopping(EARLY_STOPPING_PATIENCE, MIN_DELTA)
    best_state, best_epoch, history = None, 0, []

    epoch_iterator = tqdm(range(1, max_epochs + 1), desc=model_name)
    for epoch in epoch_iterator:
        started = time.perf_counter()
        train_loss = train_one_epoch(
            model_name, model, loss_function, loaders["train"], optimizer, scaler, epoch
        )
        metrics = evaluate_retrieval(model, loaders["gallery"], loaders["query"])
        score = metrics["mAP@10"]
        scheduler.step(score)

        if stopper.update(score):
            best_state, best_epoch = cpu_state_dict(model), epoch
        history.append({
            "epoch": epoch, "train_loss": train_loss, **metrics,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "seconds": time.perf_counter() - started,
        })
        epoch_iterator.set_postfix(loss=f"{train_loss:.4f}", map10=f"{score:.4f}")

        if trial is not None:
            trial.report(score, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if stopper.should_stop:
            break

    model.load_state_dict(best_state)
    return history, best_epoch, stopper.best_score


## Optuna studies

Each objective receives a fresh model initialized with the shared seed. Trial checkpoints and embeddings are not persisted.


In [ ]:
def suggest_parameters(trial, model_name):
    parameters = {
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-5, 3e-3, log=True
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay", 1e-6, 1e-3, log=True
        ),
    }

    if model_name == "triplet":
        parameters["margin"] = trial.suggest_float("margin", 0.1, 1.0, step=0.1)
    elif model_name == "supcon":
        parameters["temperature"] = trial.suggest_float(
            "temperature", 0.05, 0.20, step=0.025
        )
    elif model_name == "multi_similarity":
        parameters["alpha"] = trial.suggest_float("alpha", 1.0, 4.0)
        parameters["beta"] = trial.suggest_float("beta", 20.0, 80.0)
        parameters["base"] = trial.suggest_float("base", 0.3, 0.7)
    elif model_name == "arcface":
        parameters["margin"] = trial.suggest_float("margin", 0.1, 0.5)
        parameters["scale"] = trial.suggest_categorical(
            "scale", [16, 32, 48, 64]
        )
    return parameters


In [ ]:
def make_objective(model_name):
    def objective(trial):
        parameters = suggest_parameters(trial, model_name)
        model, loss_function = build_model_and_loss(model_name, parameters)
        model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
        loaders = make_model_loaders(model_name)

        try:
            _, _, best_score = fit_model(
                model_name,
                model,
                loss_function,
                loaders,
                parameters,
                TUNING_EPOCHS,
                trial=trial,
            )
            return best_score
        finally:
            del model, loss_function, loaders
            if AMP_ENABLED:
                torch.cuda.empty_cache()

    return objective


In [ ]:
def study_storage_url():
    return f"sqlite:///{OPTUNA_DB.resolve().as_posix()}"


def run_study(model_name):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    study = optuna.create_study(
        study_name=f"task4_{model_name}",
        storage=study_storage_url(),
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )
    remaining_trials = max(0, N_TRIALS - len(study.trials))
    if remaining_trials:
        study.optimize(make_objective(model_name), n_trials=remaining_trials)
    print(model_name, "best mAP@10:", study.best_value)
    print(model_name, "best parameters:", study.best_params)
    return study


def load_existing_study(model_name):
    if not OPTUNA_DB.exists():
        return None
    try:
        study = optuna.load_study(
            study_name=f"task4_{model_name}",
            storage=study_storage_url(),
        )
        completed = [
            trial for trial in study.trials
            if trial.state == optuna.trial.TrialState.COMPLETE
        ]
        return study if completed else None
    except KeyError:
        return None


### Run the five tuning studies

Set `RUN_OPTUNA = True` in the controls cell to execute or resume these studies.


In [ ]:
studies = {
    name: run_study(name) if RUN_OPTUNA else load_existing_study(name)
    for name in MODEL_NAMES
}


## Final training and self-contained checkpoints

Final models are reset and trained from scratch using their study's best parameters. A single loop preserves separate checkpoints and progress for every model.


In [ ]:
def checkpoint_payload(model_name, model, study, best_epoch, best_score):
    parameters = dict(study.best_params)
    preprocessing = (
        cae_preprocessing_config
        if model_name == "cae"
        else metric_preprocessing_config
    )
    model_config = {
        "model_name": model_name,
        "backbone": "resnet18",
        "pretrained": False,
        "input_size": list(INPUT_SIZE),
        "embedding_dimension": 512,
        "class_count": len(classes),
    }
    return {
        "model_name": model_name,
        "model_state_dict": cpu_state_dict(model),
        "model_config": model_config,
        "best_params": parameters,
        "best_epoch": best_epoch,
        "best_val_map_at_10": best_score,
        "preprocessing_config": preprocessing,
        "label_to_index": label_to_index,
        "gallery_scope": "eligible_inner_training",
        "random_seed": SEED,
    }


In [ ]:
def train_final_model(model_name, study):
    if study is None:
        print(f"Skipping {model_name}: no completed Optuna study")
        return None

    parameters = dict(study.best_params)
    model, loss_function = build_model_and_loss(model_name, parameters)
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loaders = make_model_loaders(model_name)

    history, best_epoch, best_score = fit_model(
        model_name,
        model,
        loss_function,
        loaders,
        parameters,
        FINAL_EPOCHS,
    )
    model_directory = ARTIFACT_DIR / model_name
    model_directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = model_directory / "best.pt"
    torch.save(
        checkpoint_payload(
            model_name, model, study, best_epoch, best_score
        ),
        checkpoint_path,
    )
    print(f"Saved {model_name} checkpoint: {checkpoint_path}")
    del model, loss_function, loaders
    if AMP_ENABLED:
        torch.cuda.empty_cache()
    return {"history": history, "checkpoint": checkpoint_path}


In [ ]:
final_runs = {}
if RUN_FINAL_TRAINING:
    for model_name in MODEL_NAMES:
        final_runs[model_name] = train_final_model(
            model_name,
            studies[model_name],
        )


## Sparse k-reciprocal re-ranking

This candidate-limited implementation builds reciprocal neighbourhoods from the gallery FAISS index, avoiding a dense all-pairs distance matrix.


In [ ]:
def gallery_reciprocal_sets(gallery_embeddings, index, k1):
    search_k = min(k1 + 1, len(gallery_embeddings))
    similarities, neighbours = index.search(gallery_embeddings, search_k)
    base_sets = []

    for gallery_index, row in enumerate(neighbours):
        forward = [int(item) for item in row if item != gallery_index][:k1]
        reciprocal = {
            candidate
            for candidate in forward
            if gallery_index in neighbours[candidate, 1:search_k]
        }
        base_sets.append(reciprocal)

    expanded_sets = []
    half_size = max(1, k1 // 2)
    for reciprocal in base_sets:
        expanded = set(reciprocal)
        for candidate in list(reciprocal):
            candidate_set = set(
                sorted(base_sets[candidate])[:half_size]
            )
            overlap = len(candidate_set & reciprocal)
            if candidate_set and overlap >= (2 * len(candidate_set) / 3):
                expanded.update(candidate_set)
        expanded_sets.append(expanded)

    thresholds = similarities[:, -1]
    return expanded_sets, thresholds


In [ ]:
def k_reciprocal_rerank(
    query_embeddings,
    gallery_embeddings,
    index,
    k1=20,
    k2=6,
    blend=0.3,
    maximum_k=10,
    candidate_depth=100,
):
    search_depth = min(candidate_depth, len(gallery_embeddings))
    initial_scores, candidates = index.search(query_embeddings, search_depth)
    gallery_sets, gallery_thresholds = gallery_reciprocal_sets(
        gallery_embeddings, index, k1
    )
    reranked_rows = []

    for scores, row in zip(initial_scores, candidates):
        forward_count = min(k1, len(row))
        forward = row[:forward_count]
        query_set = {
            int(candidate)
            for score, candidate in zip(scores[:forward_count], forward)
            if score >= gallery_thresholds[candidate]
        }
        for candidate in row[: min(k2, len(row))]:
            query_set.update(gallery_sets[int(candidate)])

        combined_distances = []
        for score, candidate in zip(scores, row):
            gallery_set = gallery_sets[int(candidate)]
            union = query_set | gallery_set
            intersection = query_set & gallery_set
            jaccard = 1.0 - len(intersection) / max(1, len(union))
            original_distance = 1.0 - float(score)
            distance = blend * original_distance + (1.0 - blend) * jaccard
            combined_distances.append(distance)

        ordering = np.argsort(combined_distances)[:maximum_k]
        reranked_rows.append(row[ordering])

    return np.asarray(reranked_rows, dtype=np.int64)


## Gallery export and final validation results

Each model saves checkpoint-derived gallery embeddings and IDs. The exact inner-product FAISS index is rebuilt in memory when needed.


In [ ]:
def load_checkpoint_model(model_name):
    checkpoint_path = ARTIFACT_DIR / model_name / "best.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model, _ = build_model_and_loss(model_name, checkpoint["best_params"])
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.to(DEVICE).eval(), checkpoint


In [ ]:
def save_gallery_artifacts(model_name, embeddings, ids):
    model_directory = ARTIFACT_DIR / model_name
    model_directory.mkdir(parents=True, exist_ok=True)
    np.save(model_directory / "gallery_embeddings.npy", embeddings)
    np.save(model_directory / "gallery_ids.npy", ids)
    print(f"Saved {model_name} gallery rows: {len(ids):,}")


def load_gallery_artifacts(model_name):
    model_directory = ARTIFACT_DIR / model_name
    embeddings = np.load(model_directory / "gallery_embeddings.npy").astype("float32")
    ids = np.load(model_directory / "gallery_ids.npy").astype("int64")
    label_lookup = outer_train_df.set_index("id")[LABEL_COLUMN].map(label_to_index)
    labels = pd.Series(ids).map(label_lookup).to_numpy(dtype="int64")
    return embeddings, ids, labels, build_faiss_index(embeddings)


In [ ]:
def export_and_evaluate_model(model_name):
    model, _ = load_checkpoint_model(model_name)
    gallery_loader = make_evaluation_loader(train_df, model_name)
    query_loader = make_evaluation_loader(val_df, model_name)

    gallery_embeddings, gallery_ids, gallery_labels = extract_embeddings(
        model, gallery_loader
    )
    query_embeddings, _, query_labels = extract_embeddings(model, query_loader)
    _, base_indices, index = search_cosine(
        query_embeddings, gallery_embeddings, maximum_k=10
    )
    reranked_indices = k_reciprocal_rerank(
        query_embeddings, gallery_embeddings, index,
        k1=20, k2=6, blend=0.3, maximum_k=10,
    )

    expected_ids = train_df.sort_values("id")["id"].to_numpy()
    if not np.array_equal(gallery_ids, expected_ids):
        raise ValueError("Gallery embedding and ID order do not match")
    save_gallery_artifacts(model_name, gallery_embeddings, gallery_ids)

    loaded_embeddings, loaded_ids, _, loaded_index = load_gallery_artifacts(model_name)
    search_k = min(10, len(loaded_embeddings))
    _, loaded_indices = loaded_index.search(query_embeddings, search_k)
    if not np.array_equal(base_indices, loaded_indices):
        raise ValueError("Rebuilt FAISS index changed neighbour results")
    if not np.array_equal(gallery_ids, loaded_ids):
        raise ValueError("Reloaded gallery IDs changed order")

    results = {
        "base": retrieval_metrics(base_indices, query_labels, gallery_labels),
        "reranked": retrieval_metrics(
            reranked_indices, query_labels, gallery_labels
        ),
    }
    del model
    if AMP_ENABLED:
        torch.cuda.empty_cache()
    return results


In [ ]:
final_results = {}
if RUN_GALLERY_EXPORT:
    for model_name in MODEL_NAMES:
        final_results[model_name] = export_and_evaluate_model(model_name)


## Comparison table and plots

Results stay in notebook output rather than creating separate history, metrics, or metadata JSON files.


In [ ]:
def comparison_frame(results):
    rows = []
    for model_name, variants in results.items():
        for ranking_method, metrics in variants.items():
            rows.append({
                "model": model_name,
                "ranking": ranking_method,
                **metrics,
            })
    return pd.DataFrame(rows)


comparison_df = comparison_frame(final_results)
display(comparison_df)

if not comparison_df.empty:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=comparison_df, x="model", y="mAP@10", hue="ranking")
    plt.title("Task 4 validation retrieval")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


## Optional smoke checks

This performs one optimization batch for every model and prints diagnostic values. It does not access the outer holdout.


In [ ]:
def smoke_training_step(model_name):
    defaults = {
        "learning_rate": 1e-4, "weight_decay": 1e-5,
        "margin": 0.3, "temperature": 0.1,
        "alpha": 2.0, "beta": 50.0, "base": 0.5, "scale": 32,
    }
    model, loss_function = build_model_and_loss(model_name, defaults)
    model, loss_function = model.to(DEVICE), loss_function.to(DEVICE)
    loader = make_training_loader(model_name)
    batch = next(iter(loader))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    optimizer.zero_grad(set_to_none=True)
    loss = compute_training_loss(model_name, model, loss_function, batch)
    loss.backward()
    optimizer.step()

    raw_images = batch["image"]
    sample_images = raw_images[0] if isinstance(raw_images, (list, tuple)) else raw_images
    with torch.no_grad():
        embeddings = retrieval_embeddings(model, sample_images.to(DEVICE))
    mean_norm = embeddings.norm(dim=1).mean().item()
    print(model_name, "batch:", len(batch["label"]), "loss:", loss.detach().item())
    print("Embedding:", tuple(embeddings.shape), "mean norm:", round(mean_norm, 4))
    del model, loss_function, loader
    if AMP_ENABLED:
        torch.cuda.empty_cache()


In [ ]:
if RUN_SMOKE_TESTS:
    for smoke_model_name in MODEL_NAMES:
        smoke_training_step(smoke_model_name)
else:
    print("Smoke tests disabled. Set RUN_SMOKE_TESTS = True to run them.")


## Artifact summary

After all stages finish, each model directory contains only:

- `best.pt`
- `gallery_embeddings.npy`
- `gallery_ids.npy`

Shared artifacts contain the inner split, label encoder, preprocessing configurations, and resumable Optuna database. `IndexFlatIP` is rebuilt from the saved embeddings, and no holdout-test artifact is produced here.
